> **Prerequisite:** Run earlier notebooks in numeric order in the **same kernel session**.

# 01 — Statistical models

Data loading, OLS, and moderated mediation (Hayes Model 14). No plotting code here.

In [1]:
"""Salience network topology and moderated mediation — data loading."""
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Shared network order (Δ-modularity columns in load_analysis_df; reused in figures/stats).
NET_ORDER = ["Cont", "Default", "DorsAttn", "SalVentAttn", "SomMot", "Vis"]
RT_DERIV_COLS = [
    "rt_duration_diff__long-short",
    "rt_transition_diff__flexibility_stability",
]


def _coerce_analysis_numeric(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["subject"] = out["subject"].astype(str)
    for col in [c for c in out.columns if c != "subject"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def _validate_analysis_df(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        raise ValueError(
            f"Analysis table has no participant rows ({ANALYSIS_CSV}).\n"
            "Populate data/analysis_input.csv with subject data "
            "(see data/README.md)."
        )
    return df


def _add_delta_modularity_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for net in NET_ORDER:
        col = f"global_modularity_minus_lesion_net_{net}_global_modularity_auc"
        if col in out.columns:
            continue
        intact = "global_modularity_auc"
        lesioned = f"lesion_net_{net}_global_modularity_auc"
        if intact not in out.columns or lesioned not in out.columns:
            raise KeyError(f"Cannot compute {col}: missing {intact} or {lesioned}")
        out[col] = out[intact] - out[lesioned]
    return out


# ── paths (notebook package root) ────────────────────────────────────
def _resolve_package_root() -> Path:
    """Locate mediate_moderate_lesion_analysis/ regardless of notebook cwd."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "analysis_input.csv").is_file():
            return candidate
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
        nested = candidate / "mediate_moderate_lesion_analysis"
        if (nested / "data").is_dir() and (nested / "notebooks").is_dir():
            return nested
    if start.name == "notebooks" and (start.parent / "data").is_dir():
        return start.parent
    raise FileNotFoundError(
        "Could not locate mediate_moderate_lesion_analysis package root "
        "(expected data/ and notebooks/ under the same directory). "
        f"Started search from: {start}"
    )


NOTEBOOK_DIR = Path.cwd()
MEDIATE_MODERATE_LESION_ROOT = _resolve_package_root()

DATA_DIR = MEDIATE_MODERATE_LESION_ROOT / "data"
FIGURES_DIR = MEDIATE_MODERATE_LESION_ROOT / "figures"
OUT_DIR = FIGURES_DIR

ANALYSIS_CSV = DATA_DIR / "analysis_input.csv"
FC_NPZ = DATA_DIR / "fc_display_matrices.npz"
FC_FALLBACK_NPZ = DATA_DIR / "per_subject_fc.npz"

# backfill_analysis_input.py lives at package root (parent of notebooks/)
import sys

if str(MEDIATE_MODERATE_LESION_ROOT) not in sys.path:
    sys.path.insert(0, str(MEDIATE_MODERATE_LESION_ROOT))


def _resolve_analysis_csv() -> Path:
    if ANALYSIS_CSV.is_file():
        return ANALYSIS_CSV
    raise FileNotFoundError(
        f"Analysis CSV not found: {ANALYSIS_CSV}\n"
        "Place de-identified analysis_input.csv in the data/ folder.\n"
        "See data/README.md for required columns."
    )


def _search_roots(start: Path, *, max_depth: int = 8) -> list[Path]:
    roots: list[Path] = []
    p = start.resolve()
    for _ in range(max_depth):
        if p not in roots:
            roots.append(p)
        if p.parent == p:
            break
        p = p.parent
    return roots


def _resolve_fc_npz() -> Path:
    for candidate in (FC_NPZ, FC_FALLBACK_NPZ):
        if candidate.is_file():
            return candidate
    rel_candidates = [
        "data/fc_display_matrices.npz",
        "data/per_subject_fc.npz",
        "BayesianNew/per_subject_FC.npz",
        "BayesianNew/data/per_subject_FC.npz",
        "per_subject_FC.npz",
    ]
    for root in _search_roots(MEDIATE_MODERATE_LESION_ROOT):
        for rel in rel_candidates:
            path = (root / rel).resolve()
            if path.is_file():
                return path
    raise FileNotFoundError(
        "FC matrix file not found. Provide one of:\n"
        f"  - data/{FC_NPZ.name} (recommended: group-mean display matrices)\n"
        f"  - data/{FC_FALLBACK_NPZ.name} (de-identified per-subject FC)\n"
        "  - or place per_subject_FC.npz upstream (e.g. BayesianNew/)"
    )




def _missing_rt_derivatives(df: pd.DataFrame) -> list[str]:
    return [c for c in RT_DERIV_COLS if c not in df.columns or not df[c].notna().any()]


def load_analysis_df() -> pd.DataFrame:
    """Load de-identified analysis dataset from analysis_input.csv."""
    csv_path = _resolve_analysis_csv()
    uni = pd.read_csv(csv_path)
    uni = _validate_analysis_df(_coerce_analysis_numeric(uni))

    if _missing_rt_derivatives(uni):
        from backfill_analysis_input import backfill_rt_columns

        backfill_rt_columns(csv_path)
        uni = pd.read_csv(csv_path)
        uni = _validate_analysis_df(_coerce_analysis_numeric(uni))
        still_missing = _missing_rt_derivatives(uni)
        if still_missing:
            raise ValueError(
                "RT derivative columns still missing after backfill: "
                f"{still_missing}. Run: python backfill_analysis_input.py"
            )

    return _add_delta_modularity_columns(uni)


In [2]:
df_preview = load_analysis_df()
print(f"Rows: {len(df_preview):,} | Subjects: {df_preview['subject'].nunique()}")
df_preview.filter(regex='modularity|participation|rt_').head(3)

Rows: 43 | Subjects: 43


,global_modularity_auc,lesion_net_Cont_global_modularity_auc,lesion_net_Default_global_modularity_auc,lesion_net_DorsAttn_global_modularity_auc,lesion_net_SalVentAttn_global_modularity_auc,lesion_net_SomMot_global_modularity_auc,lesion_net_Vis_global_modularity_auc,net_Cont_participation_coefficient_auc,net_Default_participation_coefficient_auc,net_DorsAttn_participation_coefficient_auc,...,net_SomMot_participation_coefficient_auc,net_Vis_participation_coefficient_auc,rt_duration_diff__long-short,rt_transition_diff__flexibility_stability,global_modularity_minus_lesion_net_Cont_global_modularity_auc,global_modularity_minus_lesion_net_Default_global_modularity_auc,global_modularity_minus_lesion_net_DorsAttn_global_modularity_auc,global_modularity_minus_lesion_net_SalVentAttn_global_modularity_auc,global_modularity_minus_lesion_net_SomMot_global_modularity_auc,global_modularity_minus_lesion_net_Vis_global_modularity_auc
0,0.129393,0.129518,0.127555,0.115916,0.129365,0.127987,0.133480,0.117502,0.095691,0.147873,...,0.095552,0.116733,90.897059,282.947203,-0.000125,0.001837,0.013476,0.000028,0.001405,-0.004087
1,0.126074,0.129550,0.124764,0.120693,0.124896,0.132151,0.125798,0.112315,0.065244,0.129264,...,0.089631,0.077236,178.754067,135.928571,-0.003476,0.001309,0.005380,0.001178,-0.006077,0.000275
2,0.132201,0.127738,0.130335,0.132338,0.124744,0.115125,0.131380,0.083136,0.072254,0.120561,...,0.097110,0.083201,146.105263,293.222906,0.004463,0.001866,-0.000137,0.007457,0.017076,0.000821


In [3]:
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from statsmodels.api import OLS, add_constant

# Shared with the setup block in 01_statistics.ipynb (same kernel).
if "NET_ORDER" not in globals():
    NET_ORDER = ["Cont", "Default", "DorsAttn", "SalVentAttn", "SomMot", "Vis"]

PC_COLS = [f"net_{n}_participation_coefficient_auc" for n in NET_ORDER]

def zscore_series(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std(ddof=1)


def fit_ols_network_pc(df: pd.DataFrame):
    sub = df[PC_COLS + ["global_modularity_auc"]].dropna().copy()
    for c in sub.columns:
        sub[c] = zscore_series(sub[c])
    x = add_constant(sub[PC_COLS])
    model = OLS(sub["global_modularity_auc"], x).fit()
    return model


def fit_moderated_mediation(df: pd.DataFrame) -> dict:
    cols = [
        "net_SalVentAttn_participation_coefficient_auc",
        "global_modularity_auc",
        "rt_duration_diff__long-short",
        "rt_transition_diff__flexibility_stability",
    ]
    sub = df[cols].dropna().copy()
    for c in cols:
        sub[c] = zscore_series(sub[c])
    x, m, w, y = (sub[c].values for c in cols)
    n = len(x)
    m_m = OLS(m, add_constant(x)).fit()
    a1 = float(m_m.params[1])
    p_a1 = float(m_m.pvalues[1])
    ci_a1 = np.asarray(m_m.conf_int())[1].tolist()
    design_y = np.column_stack([np.ones(n), m, w, m * w, x])
    m_y = OLS(y, design_y).fit()
    b1, _, b3 = (float(m_y.params[i]) for i in range(1, 4))
    p_b1, _, p_b3 = (float(m_y.pvalues[i]) for i in range(1, 4))
    imm = a1 * b3
    rs = np.random.RandomState(42)
    boot_imm = []
    boot_ind_high = []
    boot_ind_low = []
    for _ in range(5000):
        idx = rs.randint(0, n, n)
        try:
            ab = OLS(m[idx], add_constant(x[idx])).fit().params[1]
            gp = OLS(
                y[idx],
                add_constant(np.column_stack([m[idx], w[idx], m[idx] * w[idx], x[idx]])),
            ).fit().params
            bb1, bb3 = float(gp[1]), float(gp[3])
            boot_imm.append(ab * bb3)
            boot_ind_high.append(ab * (bb1 - bb3))
            boot_ind_low.append(ab * (bb1 + bb3))
        except Exception:
            pass
    imm_ci = np.percentile(boot_imm, [2.5, 97.5]) if boot_imm else [np.nan, np.nan]
    ind_high = a1 * (b1 - b3)
    ind_low = a1 * (b1 + b3)
    ind_high_ci = (
        np.percentile(boot_ind_high, [2.5, 97.5]).tolist() if boot_ind_high else [np.nan, np.nan]
    )
    ind_low_ci = (
        np.percentile(boot_ind_low, [2.5, 97.5]).tolist() if boot_ind_low else [np.nan, np.nan]
    )
    return {
        "n": n,
        "a1": a1,
        "p_a1": p_a1,
        "ci_a1": ci_a1,
        "b1": b1,
        "b3": b3,
        "p_b1": p_b1,
        "p_b3": p_b3,
        "imm": imm,
        "imm_ci": imm_ci,
        "ind_high": ind_high,
        "ind_low": ind_low,
        "ind_high_ci": ind_high_ci,
        "ind_low_ci": ind_low_ci,
    }


MODERATED_MEDIATION_COLS = [
    "net_SalVentAttn_participation_coefficient_auc",
    "global_modularity_auc",
    "rt_duration_diff__long-short",
    "rt_transition_diff__flexibility_stability",
]


